# Wildlife Re-Identification — hyenas & leopards (MAP@5)

**Pipeline**
1. **Calibrated validation**: 5 splits built from train that mimic the test set (unseen individuals + new "encounters" of known ones).
2. **Retrieval**: MiewID descriptors → centering + L2 → cosine kNN, plus a PCA(256) → **LDA** projection (linear metric learning). Top-20 candidate identities per query (same species only).
3. **SIFT geometric verification** on the *provided* keypoints: RootSIFT → 2-NN → Lowe ratio test → one-to-one → **RANSAC** (homography + similarity). Up to 20 gallery images for each of the top-5 candidates, 2 images for ranks 6–10.
4. **Viewpoint** compatibility (left/right flank) as soft features.
5. **LightGBM** round 1: P(match | query, candidate) and P(new individual | query).
6. **Round 2 (transductive)**: the query set's own kNN graph. Each query sees what round 1 says about its nearest query images. Then refit.
7. **Stage 3 ranker**: a LightGBM LambdaRank over {candidates + `new_individual`} per query, blended with the stage-2 probabilities (weight chosen by CV). Always 5 guesses.

**Rules compliance.** Only the provided features (MiewID, SIFT keypoints/descriptors, species, viewpoint) are used. No images are read and no pretrained models are used. Methods are all on the allowed list: PCA, LDA, kNN/distance retrieval, gradient boosting, stacking, ratio test + RANSAC. Libraries used: numpy, pandas, scipy, scikit-learn, lightgbm, opencv-python (keypoint matching/verification only). No deep learning, AutoML or LLM-generated labels.

In [1]:
import os, glob, time, pickle, warnings
import numpy as np, pandas as pd
import cv2, lightgbm as lgb
from scipy.sparse.csgraph import connected_components
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from concurrent.futures import ThreadPoolExecutor
warnings.filterwarnings("ignore")
cv2.setNumThreads(1)

SEED = 0
np.random.seed(SEED)
NEW = "new_individual"

# ---------------- config ----------------
N_SPLITS   = 16                      # calibrated validation splits (also the training data for stage-2 models)
EVAL_SPLITS = list(range(8))         # splits used to report CV of stages 2-3 (saves time; all 16 are used for training)
SPLIT_CFG  = dict(enc_thr=0.8, new_frac=0.1, q_frac=0.4, per_enc=1)
K_CAND, K_SIFT, IMG_PER_ID, DEEP_R, DEEP_IMG, K_QQ = 20, 10, 2, 5, 20, 5
LGB_SEEDS  = [0, 1, 2]
W_GRID     = [0.8, 1.0, 1.25]        # weight of P(new) when ranking; chosen by CV
N_THREADS  = os.cpu_count() or 4

# ---------------- paths (Kaggle or local) ----------------
def find_data_dir():
    roots = ["/kaggle/input", os.path.expanduser("~/Downloads/projectwild3"), ".", ".."]
    for r in roots:
        hits = glob.glob(os.path.join(r, "**", "miewid_features.npy"), recursive=True)
        if hits: return os.path.dirname(hits[0])
    raise FileNotFoundError("miewid_features.npy not found")
def find_sift(name):
    for r in ["/kaggle/input", os.path.expanduser("~/Downloads/projectwild3"), ".", ".."]:
        hits = glob.glob(os.path.join(r, "**", name), recursive=True)
        if hits: return hits[0]
    raise FileNotFoundError(name)
DATA = find_data_dir()
SIFT_TR, SIFT_TE = find_sift("sift_features_train.npz"), find_sift("sift_features_test.npz")
WORK = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."
def log(msg):
    line = time.strftime("%H:%M:%S ") + msg
    print(line, flush=True)
    with open(f"{WORK}/progress.log", "a") as fh: fh.write(line + chr(10))
print("data:", DATA, "\nsift:", SIFT_TR, SIFT_TE, "\nthreads:", N_THREADS)

data: <data_dir>/reid_student_data_main 
sift: <data_dir>/sift_features_train.npz <data_dir>/sift_features_test.npz 
threads: 20


## 1. Data loading & integrity checks

In [2]:
train = pd.read_csv(f"{DATA}/train.csv"); test = pd.read_csv(f"{DATA}/test.csv")
mid = pd.read_csv(f"{DATA}/miewid_ids.csv").id.values
X = np.load(f"{DATA}/miewid_features.npy").astype(np.float32)
pos = {k: i for i, k in enumerate(mid)}
assert X.shape == (len(mid), 2152) and not np.isnan(X).any()
assert set(train.id) | set(test.id) == set(mid) and len(set(mid)) == len(mid)
Xtr, Xte = X[[pos[i] for i in train.id]], X[[pos[i] for i in test.id]]
for d in (train, test): d["viewpoint"] = d["viewpoint"].fillna("unknown")
train = train.reset_index(drop=True); test = test.reset_index(drop=True)

def l2n(A): return A / np.linalg.norm(A, axis=1, keepdims=True).clip(1e-12)

print(f"train {len(train)}  test {len(test)}")
for s in ["hyena", "leopard"]:
    c = train[train.species == s].identity.value_counts()
    print(f"{s:8s} ids {len(c):4d} | train imgs {c.sum():5d} | test imgs {(test.species==s).sum():5d} | singletons {(c==1).sum()} | median imgs/id {int(c.median())}")

train 7163  test 2791
hyena    ids  205 | train imgs  1912 | test imgs  1217 | singletons 3 | median imgs/id 6
leopard  ids  345 | train imgs  5251 | test imgs  1574 | singletons 99 | median imgs/id 3


In [3]:
# SIFT: works with real .npz files (Kaggle) or extracted folders of <id>_xy.npy / <id>_desc.npy
def load_sift(path, ids):
    out = {}
    if os.path.isdir(path):
        for i in ids:
            out[i] = (np.load(f"{path}/{i}_xy.npy").astype(np.float32).reshape(-1, 2),
                      np.load(f"{path}/{i}_desc.npy").astype(np.uint8).reshape(-1, 128))
    else:
        z = np.load(path)
        for i in ids:
            out[i] = (z[f"{i}_xy"].astype(np.float32).reshape(-1, 2), z[f"{i}_desc"].astype(np.uint8).reshape(-1, 128))
    return out
t = time.time()
SIFT = load_sift(SIFT_TR, train.id); SIFT.update(load_sift(SIFT_TE, test.id))
nkp = {k: len(v[0]) for k, v in SIFT.items()}
assert all(len(v[0]) == len(v[1]) for v in SIFT.values())
kp = np.array(list(nkp.values()))
print(f"SIFT loaded in {time.time()-t:.0f}s | images {len(SIFT)} | zero-kp {(kp==0).sum()} | <50 kp {(kp<50).mean():.3f} | <500 kp {(kp<500).mean():.3f}")

SIFT loaded in 421s | images 9954 | zero-kp 96 | <50 kp 0.049 | <500 kp 0.193


## 2. Classical SIFT matching (ratio test + RANSAC)
RootSIFT (L1-normalise, then square root) is a fixed transform of the given descriptors. The RANSAC seed is fixed per pair, so results are reproducible with any thread count. Results are cached on disk.

In [4]:
def root_desc(i):
    d = SIFT[i][1].astype(np.float32)
    if len(d): d = np.sqrt(d / np.maximum(d.sum(1, keepdims=True), 1e-7))
    return d

def sift_match(a, b, ratio=0.8):
    # returns (n ratio-test matches, homography inliers, similarity inliers, inlier fraction)
    xa, xb = SIFT[a][0], SIFT[b][0]; da, db = root_desc(a), root_desc(b)
    if len(da) < 3 or len(db) < 3: return (0, 0, 0, 0.0)
    cv2.setRNGSeed(12345)
    m = cv2.BFMatcher(cv2.NORM_L2).knnMatch(da, db, k=2)
    best = {}
    for x in m:
        if len(x) == 2 and x[0].distance < ratio * x[1].distance:
            g = x[0]
            if g.trainIdx not in best or g.distance < best[g.trainIdx].distance: best[g.trainIdx] = g
    good = list(best.values()); n = len(good)
    if n < 4: return (n, 0, 0, 0.0)
    p = xa[[g.queryIdx for g in good]]; q = xb[[g.trainIdx for g in good]]
    _, mh = cv2.findHomography(p, q, cv2.RANSAC, 8.0, maxIters=2000, confidence=0.995)
    nh = int(mh.sum()) if mh is not None else 0
    _, ma = cv2.estimateAffinePartial2D(p, q, method=cv2.RANSAC, ransacReprojThreshold=8.0, maxIters=2000)
    na = int(ma.sum()) if ma is not None else 0
    return (n, nh, na, max(nh, na) / n)

class SiftCache:
    path = f"{WORK}/sift_pair_cache.pkl"
    def __init__(self):
        self.d = pickle.load(open(self.path, "rb")) if os.path.exists(self.path) else {}
    def fill(self, pairs):
        need = sorted({tuple(sorted(p)) for p in pairs} - self.d.keys())
        if not need: return
        t = time.time(); log(f"   SIFT: matching {len(need)} pairs")
        with ThreadPoolExecutor(N_THREADS) as ex:
            res = list(ex.map(lambda p: sift_match(*p), need, chunksize=256))
        self.d.update(zip(need, res))
        pickle.dump(self.d, open(self.path, "wb"), protocol=4)
        log(f"   SIFT: matched {len(need)} pairs in {time.time()-t:.0f}s (cache {len(self.d)})")
    def __call__(self, a, b): return self.d[tuple(sorted((a, b)))]
SC = SiftCache()

## 3. Validation design: mimic the test set
There is no validation file, so we simulate the test from train:
* **Encounters**: within each identity, connected components of images with cosine > 0.8 (near-duplicate shots).
* Hold out **10 % of identities** entirely. Their images become `new_individual` queries.
* For the remaining identities, hold out **~40 % of encounters**, always keeping at least one in the gallery.
* Keep **one query image per held-out encounter**, because test images rarely have near-duplicate siblings among other test images.

Calibrated on two test statistics at once: similarity to the nearest training image, and similarity to the nearest *other test* images. An earlier design matched only the first statistic; it gave CV 0.91 against a public LB of 0.80. With this design the old model scores 0.85 locally, much closer to that leaderboard result.

In [5]:
A = l2n(Xtr)
def encounters(thr):
    enc = np.zeros(len(train), int); nxt = 0
    for _, g in train.groupby("identity"):
        idx = g.index.values; n, lab = connected_components(A[idx] @ A[idx].T > thr, directed=False)
        enc[idx] = lab + nxt; nxt += n
    return enc

def make_split(enc, seed, new_frac, q_frac, per_enc):
    rng = np.random.RandomState(seed); is_q = np.zeros(len(train), bool)
    for s in ["hyena", "leopard"]:
        m = (train.species == s).values
        idents = train.identity[m].unique()
        held = set(rng.choice(idents, int(round(len(idents) * new_frac)), replace=False))
        is_q |= m & train.identity.isin(held).values
        for _, g in train[m & ~train.identity.isin(held).values].groupby("identity"):
            e = np.unique(enc[g.index.values])
            if len(e) < 2: continue
            pick = [x for x in e if rng.rand() < q_frac]
            if len(pick) == len(e): pick = pick[1:]
            is_q[g.index.values[np.isin(enc[g.index.values], pick)]] = True
    gal = ~is_q
    # test images rarely have near-duplicate siblings among other test images -> keep per_enc query image(s) per encounter
    rng2 = np.random.RandomState(1000 + seed); idx = np.where(is_q)[0]; keep = np.zeros_like(is_q)
    for e in np.unique(enc[idx]):
        m = idx[enc[idx] == e]; keep[rng2.choice(m, min(per_enc, len(m)), replace=False)] = True
    gal_ids = set(train.identity[gal])
    truth = np.array([t if t in gal_ids else NEW for t in train.identity[keep]])
    return gal, keep, truth

ENC = encounters(SPLIT_CFG["enc_thr"])
SPLITS = [make_split(ENC, s, SPLIT_CFG["new_frac"], SPLIT_CFG["q_frac"], SPLIT_CFG["per_enc"]) for s in range(N_SPLITS)]

def sim_profile(Xg, gs, Xq, qs):
    # (a) query -> gallery top similarity, (b) query -> other queries mean top-5 similarity (centered cosine, same species)
    mu = Xg.mean(0); G = l2n(Xg - mu); Q = l2n(Xq - mu); top, qq = [], []
    for s in ["hyena", "leopard"]:
        g, q = G[gs == s], Q[qs == s]; top.append((q @ g.T).max(1))
        S = q @ q.T; np.fill_diagonal(S, -1); qq.append(np.sort(S, 1)[:, -5:].mean(1))
    return np.percentile(np.concatenate(top), [25, 50, 75]).round(3), np.percentile(np.concatenate(qq), [25, 50, 75]).round(3)
t = sim_profile(Xtr, train.species.values, Xte, test.species.values)
print(f"TEST  query->gallery p25/50/75 {t[0]} | query->query {t[1]}")
for s, (gal, q, truth) in enumerate(SPLITS):
    v = sim_profile(Xtr[gal], train.species.values[gal], Xtr[q], train.species.values[q])
    print(f"val{s}  query->gallery p25/50/75 {v[0]} | query->query {v[1]} | queries {q.sum()} new {np.mean(truth==NEW):.2f}")

TEST  query->gallery p25/50/75 [0.347 0.446 0.625] | query->query [0.314 0.388 0.501]


val0  query->gallery p25/50/75 [0.335 0.434 0.582] | query->query [0.31  0.38  0.485] | queries 2525 new 0.28
val1  query->gallery p25/50/75 [0.344 0.455 0.606] | query->query [0.313 0.381 0.472] | queries 2406 new 0.24


val2  query->gallery p25/50/75 [0.35  0.472 0.612] | query->query [0.31  0.375 0.467] | queries 2434 new 0.18
val3  query->gallery p25/50/75 [0.349 0.462 0.603] | query->query [0.312 0.383 0.484] | queries 2439 new 0.23


val4  query->gallery p25/50/75 [0.363 0.486 0.624] | query->query [0.311 0.377 0.471] | queries 2342 new 0.17
val5  query->gallery p25/50/75 [0.349 0.46  0.608] | query->query [0.309 0.377 0.472] | queries 2311 new 0.20


val6  query->gallery p25/50/75 [0.349 0.459 0.6  ] | query->query [0.32  0.39  0.481] | queries 2452 new 0.26
val7  query->gallery p25/50/75 [0.338 0.453 0.599] | query->query [0.313 0.386 0.491] | queries 2456 new 0.23


val8  query->gallery p25/50/75 [0.345 0.444 0.593] | query->query [0.311 0.386 0.487] | queries 2523 new 0.28
val9  query->gallery p25/50/75 [0.349 0.459 0.603] | query->query [0.312 0.381 0.478] | queries 2451 new 0.22


val10  query->gallery p25/50/75 [0.338 0.441 0.587] | query->query [0.317 0.393 0.491] | queries 2524 new 0.28
val11  query->gallery p25/50/75 [0.339 0.445 0.596] | query->query [0.311 0.379 0.481] | queries 2421 new 0.24


val12  query->gallery p25/50/75 [0.345 0.464 0.607] | query->query [0.306 0.375 0.472] | queries 2412 new 0.20
val13  query->gallery p25/50/75 [0.345 0.448 0.591] | query->query [0.311 0.38  0.493] | queries 2438 new 0.26


val14  query->gallery p25/50/75 [0.347 0.461 0.614] | query->query [0.311 0.382 0.477] | queries 2357 new 0.21
val15  query->gallery p25/50/75 [0.342 0.456 0.604] | query->query [0.315 0.391 0.486] | queries 2507 new 0.27


### One-vs-many keypoint matching (HotSpotter-style LNBNN)
Every RootSIFT descriptor of a query is matched against **all** training descriptors of its species (exact nearest
neighbours on a faiss-cpu IVF index; classical NN matching of the provided keypoints, no new features). Identity score =
LNBNN: for each query descriptor and each identity among its K nearest valid neighbours, add
`d_background - d_nearest_of_identity` (background = (K+1)-th neighbour). Neighbour lists are computed once; at scoring
time images outside the current gallery (and the query itself) are masked, so validation stays honest.
Leak-free A/B on the round-1 model (splits 0-7): 0.8473 -> 0.8518, better on 8/8 splits.

In [6]:
import json, faiss
faiss.omp_set_num_threads(N_THREADS)
LNB_DIR, LNB_KNN, LNB_K = f"{WORK}/lnbnn", 50, 10

def lnb_build(sp):
    gids = train.id[train.species == sp].values; qids = np.r_[gids, test.id[test.species == sp].values]
    if os.path.exists(f"{LNB_DIR}/{sp}_meta.json"):
        m = json.load(open(f"{LNB_DIR}/{sp}_meta.json"))
        if m["gallery"] == gids.tolist() and m["queries"] == qids.tolist(): return
    os.makedirs(LNB_DIR, exist_ok=True); t = time.time()
    X = np.concatenate([root_desc(g).reshape(-1, 128) for g in gids]).astype(np.float32)
    owner = np.repeat(np.arange(len(gids)), [nkp[g] for g in gids]).astype(np.int32)
    index = faiss.IndexIVFFlat(faiss.IndexFlatL2(128), 128, 2048 if sp == "hyena" else 4096)
    index.train(X[np.random.RandomState(0).choice(len(X), min(len(X), 300000), replace=False)]); index.add(X); index.nprobe = 12
    qoff = np.r_[0, np.cumsum([nkp[q] for q in qids])]
    NB = np.lib.format.open_memmap(f"{LNB_DIR}/{sp}_nb.npy", "w+", np.int32, (int(qoff[-1]), LNB_KNN))
    DI = np.lib.format.open_memmap(f"{LNB_DIR}/{sp}_d.npy", "w+", np.float16, (int(qoff[-1]), LNB_KNN))
    for b in range(0, len(qids), 400):
        e = min(b + 400, len(qids))
        if qoff[e] == qoff[b]: continue
        Q = np.concatenate([root_desc(q).reshape(-1, 128) for q in qids[b:e]]).astype(np.float32)
        d, i = index.search(Q, LNB_KNN)
        NB[qoff[b]:qoff[e]] = np.where(i >= 0, owner[np.maximum(i, 0)], -1); DI[qoff[b]:qoff[e]] = np.sqrt(np.maximum(d, 0))
    NB.flush(); DI.flush()
    json.dump({"gallery": gids.tolist(), "queries": qids.tolist(), "qoff": qoff.tolist()}, open(f"{LNB_DIR}/{sp}_meta.json", "w"))
    log(f"LNBNN neighbours for {sp}: {len(X)} gallery descriptors, {len(qids)} query images in {time.time() - t:.0f}s")

class LNBScorer:
    def __init__(self, sp):
        m = json.load(open(f"{LNB_DIR}/{sp}_meta.json"))
        self.gal = np.array(m["gallery"]); self.qpos = {q: i for i, q in enumerate(m["queries"])}; self.qoff = np.array(m["qoff"])
        self.NB = np.load(f"{LNB_DIR}/{sp}_nb.npy", mmap_mode="r"); self.DI = np.load(f"{LNB_DIR}/{sp}_d.npy", mmap_mode="r")
        lab = dict(zip(train.id, train.identity)); self.names, self.lab = np.unique([lab[g] for g in self.gal], return_inverse=True)
        self.gpos = {g: i for i, g in enumerate(self.gal)}
    def scores(self, qid, gmask):
        i = self.qpos[qid]; a, b = self.qoff[i], self.qoff[i + 1]; out = np.zeros(len(self.names), np.float32)
        if b == a: return out
        nb = np.asarray(self.NB[a:b]); d = np.asarray(self.DI[a:b], np.float32)
        ok = (nb >= 0) & gmask[np.maximum(nb, 0)]
        if qid in self.gpos: ok &= nb != self.gpos[qid]
        rank = np.cumsum(ok, 1); has_bg = rank[:, -1] >= LNB_K + 1
        bg = np.where(has_bg, d[np.arange(len(d)), np.argmax(rank >= LNB_K + 1, 1)], d.max(1))
        lab = self.lab[np.maximum(nb, 0)]; r, c = np.nonzero(ok & (rank <= LNB_K))
        _, first = np.unique(r.astype(np.int64) * len(self.names) + lab[r, c], return_index=True); r, c = r[first], c[first]
        np.add.at(out, lab[r, c], np.maximum(bg[r] - d[r, c], 0)); return out

for sp in ["hyena", "leopard"]: lnb_build(sp)
LNB = {sp: LNBScorer(sp) for sp in ["hyena", "leopard"]}

## 4. Features for (query, candidate identity) pairs and for each query
Everything is fitted on the gallery only (train-only for validation queries, full train for test).

In [7]:
SIDE = {"left": "L", "frontleft": "L", "backleft": "L", "right": "R", "frontright": "R", "backright": "R"}
def n_img(r): return DEEP_IMG if r < DEEP_R else IMG_PER_ID

def embed(Xg, yg, Xq):
    mu = Xg.mean(0)
    E1g, E1q = l2n(Xg - mu), l2n(Xq - mu)                       # centered cosine
    Gn, Qn = l2n(Xg), l2n(Xq); m = Gn.mean(0)
    p = PCA(256, random_state=0).fit(Gn - m)
    g, q = p.transform(Gn - m), p.transform(Qn - m)
    l = LDA(solver="eigen", shrinkage=0.3).fit(g, yg)            # LDA projection (linear metric learning)
    return (E1g, E1q), (l2n(l.transform(g)), l2n(l.transform(q)))

def build(gal, Xg, qry, Xq):
    pair_rows, q_rows = [], []
    for sp in ["hyena", "leopard"]:
        gm = (gal.species == sp).values; qm = (qry.species == sp).values
        G = gal[gm].reset_index(drop=True); Q = qry[qm].reset_index(drop=True)
        log(f"  build[{sp}] embed: {len(G)} gallery / {len(Q)} queries")
        (cg, cq), (lg, lq) = embed(Xg[gm], G.identity.values, Xq[qm])
        Sc = cq @ cg.T; Sl = lq @ lg.T; Sqq = cq @ cq.T; np.fill_diagonal(Sqq, -1)
        names, inv = np.unique(G.identity.values, return_inverse=True)
        nid = len(names); cnt = np.bincount(inv); members = [np.where(inv == j)[0] for j in range(nid)]
        Ic = np.full((len(Q), nid), -9, np.float32); Il = Ic.copy(); Ic3 = Ic.copy(); best_img = np.zeros((len(Q), nid), int)
        for j, cols in enumerate(members):
            sub = Sc[:, cols]; Ic[:, j] = sub.max(1); best_img[:, j] = cols[sub.argmax(1)]
            Il[:, j] = Sl[:, cols].max(1); Ic3[:, j] = -np.sort(-sub, 1)[:, :min(3, len(cols))].mean(1)
        comb = Ic + Il
        cand = np.argsort(-comb, 1)[:, :K_CAND]
        nn_q = np.argsort(-Sqq, 1)[:, :K_QQ]
        def sift_imgs(i, r, j):
            cols = members[j]; return cols[np.argsort(-Sc[i, cols])[:n_img(r)]]
        pairs = []
        for i in range(len(Q)):
            for r, j in enumerate(cand[i, :K_SIFT]): pairs += [(Q.id[i], G.id[t]) for t in sift_imgs(i, r, j)]
            pairs += [(Q.id[i], Q.id[t]) for t in nn_q[i, :2]]
        log(f"  build[{sp}] {len(pairs)} SIFT pairs needed")
        SC.fill(pairs)
        log(f"  build[{sp}] feature rows")
        LS = LNB[sp]; gmask = np.isin(LS.gal, G.id.values)
        lnb = np.stack([LS.scores(q, gmask) for q in Q.id])[:, np.searchsorted(LS.names, names)]   # queries x gallery ids
        lnb_rank = np.argsort(np.argsort(-lnb, 1), 1); lnb_mx = lnb.max(1); lnb_2 = -np.sort(-lnb, 1)[:, 1] if nid > 1 else lnb_mx
        side_g = G.viewpoint.map(SIDE).fillna("O").values; side_q = Q.viewpoint.map(SIDE).fillna("O").values
        vp_g = G.viewpoint.values; vp_q = Q.viewpoint.values
        for i in range(len(Q)):
            row = comb[i]; top1, top2 = row[cand[i, 0]], row[cand[i, 1]]
            qq_sift = [SC(Q.id[i], Q.id[t])[1] for t in nn_q[i, :2]]
            sift_best = 0; srow = {}
            for r, j in enumerate(cand[i, :K_SIFT]):
                m = np.array([SC(Q.id[i], G.id[t]) for t in sift_imgs(i, r, j)])
                hs = np.sort(m[:, 2])[::-1]
                srow[j] = (m[:, 1].max(), m[:, 2].max(), m[:, 0].max(), m[:, 3].max(), int((m[:, 2] >= 8).sum()), hs[:2].sum())
                sift_best = max(sift_best, m[:, 1].max())
            nbr = nn_q[i]
            for r, j in enumerate(cand[i]):
                bi = best_img[i, j]; cidx = members[j]
                s = srow.get(j, (-1, -1, -1, -1, -1, -1))
                sup = Sqq[i, nbr] * Ic[nbr, j]
                ok = ~(((side_g[cidx] == "L") & (side_q[i] == "R")) | ((side_g[cidx] == "R") & (side_q[i] == "L")))
                cos_side = Sc[i, cidx[ok]].max() if ok.any() else -1.0
                lda_side = Sl[i, cidx[ok]].max() if ok.any() else -1.0
                pair_rows.append(dict(
                    qid=Q.id[i], cand=names[j], species=int(sp == "leopard"), rank=r,
                    cos_max=Ic[i, j], cos_top3=Ic3[i, j], lda_max=Il[i, j], comb=row[j],
                    gap_top1=top1 - row[j], margin_next=row[j] - (row[cand[i, r + 1]] if r + 1 < K_CAND else row[j] - 0.1),
                    n_imgs=cnt[j], log_n=np.log1p(cnt[j]),
                    sift_h=s[0], sift_s=s[1], sift_ratio=s[2], sift_frac=s[3], sift_hits=s[4], sift_top2=s[5], sift_rel=s[0] - sift_best,
                    vp_same=int(vp_g[bi] == vp_q[i]), side_conflict=int({side_g[bi], side_q[i]} == {"L", "R"}),
                    frac_same_side=float(np.mean(side_g[cidx] == side_q[i])),
                    cos_side=cos_side, lda_side=lda_side, side_drop=Ic[i, j] - cos_side,
                    nbr_sup_max=sup.max(), nbr_sup_mean=sup.mean(), nbr_is_top=float(np.mean(comb[nbr].argmax(1) == j)),
                    q_kp=nkp[Q.id[i]], g_kp=nkp[G.id[bi]],
                    lnb=lnb[i, j], lnb_rank=lnb_rank[i, j], lnb_rel=lnb[i, j] / lnb_mx[i] if lnb_mx[i] > 0 else 0.0,
                    lnb_log=np.log1p(lnb[i, j])))
            q_rows.append(dict(
                qid=Q.id[i], species=int(sp == "leopard"), top1=top1, top2=top2, gap12=top1 - top2,
                cos1=Ic[i, cand[i, 0]], lda1=Il[i, cand[i, 0]], sift_best=sift_best, sift_top1=srow[cand[i, 0]][0],
                qq1=Sqq[i, nbr[0]], qq_mean=Sqq[i, nbr].mean(), qq_sift=max(qq_sift), nbr_top1=comb[nbr[0]].max(),
                nbr_gal_gap=comb[nbr[0]].max() - Sqq[i, nbr[0]], dens=np.sort(Sc[i])[-10:].mean(),
                side=int(side_q[i] == "L") - int(side_q[i] == "R"), vp_known=int(vp_q[i] != "unknown"), q_kp=nkp[Q.id[i]],
                nbr_ids=list(Q.id[nbr]), nbr_sims=list(Sqq[i, nbr]),
                lnb_max=lnb_mx[i], lnb_gap=lnb_mx[i] - lnb_2[i], lnb_log_max=np.log1p(lnb_mx[i]),
                lnb_agree=float(lnb[i].argmax() == cand[i, 0])))
    return pd.DataFrame(pair_rows), pd.DataFrame(q_rows)

In [8]:
VAL = {}
for s, (gal, q, truth) in enumerate(SPLITS):
    t = time.time()
    P, Qf = build(train[gal].reset_index(drop=True), Xtr[gal], train[q].reset_index(drop=True), Xtr[q])
    tm = dict(zip(train.id[q], truth))
    P["y"] = (P.cand == P.qid.map(tm)).astype(int); Qf["truth"] = Qf.qid.map(tm); Qf["is_new"] = (Qf.truth == NEW).astype(int)
    VAL[s] = (P, Qf)
    rec = P.groupby("qid").y.max().reindex(Qf.qid[Qf.truth != NEW]).mean()
    log(f"split {s}: {len(Qf)} queries | new {Qf.is_new.mean():.2f} | candidate recall (known) {rec:.4f} | {time.time()-t:.0f}s")
t = time.time()
PT, QT = build(train, Xtr, test, Xte)
print(f"test: {len(QT)} queries, {len(PT)} pairs | {time.time()-t:.0f}s")
# free memory: raw SIFT arrays and the pair-match cache are no longer needed (cache stays on disk)
import gc; del SIFT, SC; gc.collect()
for d in list(VAL.values()) + [(PT, QT)]:
    for df in d:
        for c in df.select_dtypes("float64").columns: df[c] = df[c].astype(np.float32)

10:13:00   build[hyena] embed: 1011 gallery / 769 queries


10:13:01   build[hyena] 40510 SIFT pairs needed


10:13:01   build[hyena] feature rows


10:13:04   build[leopard] embed: 2529 gallery / 1756 queries


10:13:07   build[leopard] 141319 SIFT pairs needed


10:13:07   build[leopard] feature rows


... (109 outputs left out)

10:16:57   build[leopard] embed: 5251 gallery / 1574 queries


10:17:01   build[leopard] 120956 SIFT pairs needed


10:17:01   build[leopard] feature rows


test: 2791 queries, 55820 pairs | 18s


## 5. Two-round LightGBM stacking

In [9]:
PAIR_F1 = ["species", "rank", "cos_max", "cos_top3", "lda_max", "comb", "gap_top1", "margin_next", "n_imgs", "log_n",
           "sift_h", "sift_s", "sift_ratio", "sift_frac", "sift_rel", "vp_same", "side_conflict", "frac_same_side",
           "nbr_sup_max", "nbr_sup_mean", "nbr_is_top", "q_kp", "g_kp", "cos_side", "lda_side", "side_drop", "sift_hits", "sift_top2",
           "lnb", "lnb_rank", "lnb_rel", "lnb_log"]
Q_F1 = ["species", "top1", "top2", "gap12", "cos1", "lda1", "sift_best", "sift_top1", "qq1", "qq_mean", "qq_sift",
        "nbr_top1", "nbr_gal_gap", "dens", "side", "vp_known", "q_kp", "lnb_max", "lnb_gap", "lnb_log_max", "lnb_agree"]
PAIR_F2 = PAIR_F1 + ["p1", "p1_rank", "p1_norm", "pn1_self", "nb_p_max", "nb_p_w", "nb0_p", "nb_cnt"]
Q_F2 = Q_F1 + ["pn1", "p1_max", "p1_2nd", "nb_pn_w", "nb0_pn", "nb_known_conf", "nb_agree"]
PARAMS = dict(objective="binary", learning_rate=0.03, num_leaves=31, min_child_samples=40, feature_fraction=0.8,
              bagging_fraction=0.8, bagging_freq=1, lambda_l2=1.0, verbose=-1, num_threads=N_THREADS)

def fit(P, Qf, pf, qf, seed=0):
    p = dict(PARAMS, seed=seed)
    return (lgb.train(p, lgb.Dataset(P[pf], P.y), 600), lgb.train(p, lgb.Dataset(Qf[qf], Qf.is_new), 400))

def fit_predict(Ptr, Qtr, Pte, Qte, pf, qf, seeds, keep=None):
    pp, pn = np.zeros(len(Pte)), np.zeros(len(Qte))
    for s in seeds:
        mp, mq = fit(Ptr, Qtr, pf, qf, s)
        if keep is not None: keep.append((mp, mq))
        pp += mp.predict(Pte[pf]) / len(seeds); pn += mq.predict(Qte[qf]) / len(seeds)
    return pp, pn

def predict_with(models, P, Qf, pf, qf):
    return (np.mean([mp.predict(P[pf]) for mp, _ in models], 0), np.mean([mq.predict(Qf[qf]) for _, mq in models], 0))

def round2_feats(P, Qf):
    # neighbour evidence from the query set's own kNN graph (no labels used)
    P = P.copy(); Qf = Qf.copy()
    P["p1_rank"] = P.groupby("qid").p1.rank(ascending=False)
    P["p1_norm"] = P.p1 / P.groupby("qid").p1.transform("sum").clip(1e-9)
    pmap = {(q, c): p for q, c, p in zip(P.qid, P.cand, P.p1)}
    pn = dict(zip(Qf.qid, Qf.pn1)); P["pn1_self"] = P.qid.map(pn)
    top = P.sort_values("p1", ascending=False).drop_duplicates("qid").set_index("qid")
    topc, topp = top.cand.to_dict(), top.p1.to_dict()
    nb = {q: (ids, np.array(s)) for q, ids, s in zip(Qf.qid, Qf.nbr_ids, Qf.nbr_sims)}
    a = np.zeros((len(P), 4), np.float32)
    for r, (q, c) in enumerate(zip(P.qid.values, P.cand.values)):
        ids, s = nb[q]; v = np.array([pmap.get((n, c), 0.0) for n in ids]); w = np.clip(s, 0, None)
        a[r] = (np.max(s * v), (w * v).sum() / max(w.sum(), 1e-6), v[0] * s[0], np.sum((v > 0.5) & (s > 0.6)))
    P[["nb_p_max", "nb_p_w", "nb0_p", "nb_cnt"]] = a
    g = P.groupby("qid").p1
    Qf["p1_max"] = Qf.qid.map(g.max()); Qf["p1_2nd"] = Qf.qid.map(g.apply(lambda x: np.sort(x.values)[-2]))
    b = np.zeros((len(Qf), 4), np.float32)
    for r, (q, ids, s) in enumerate(zip(Qf.qid, Qf.nbr_ids, Qf.nbr_sims)):
        s = np.array(s); w = np.clip(s, 0, None); pns = np.array([pn[n] for n in ids])
        conf = np.array([topp[n] for n in ids]); agree = np.array([topc[n] == topc[q] for n in ids])
        b[r] = ((w * pns).sum() / max(w.sum(), 1e-6), pns[0], np.max(s * conf), np.max(s * agree))
    Qf[["nb_pn_w", "nb0_pn", "nb_known_conf", "nb_agree"]] = b
    return P, Qf

def rank_preds(P, Qf, pcol, ncol, w):
    # sort identities by P(match) and insert new_individual where w*P(new) beats them; always 5 labels
    pn = dict(zip(Qf.qid, Qf[ncol])); out = {}
    for qid, g in P.groupby("qid", sort=False):
        s = g[pcol].values; o = np.argsort(-s); lst = []; placed = False
        for j in o[:5]:
            if not placed and s[j] < w * pn[qid]: lst.append(NEW); placed = True
            lst.append(g.cand.values[j])
        if not placed: lst.append(NEW)
        out[qid] = lst[:5]
    return out

def map5(preds, truth):
    s = 0.0
    for p, t in zip(preds, truth):
        for k, lab in enumerate(p[:5]):
            if lab == t: s += 1 / (k + 1); break
    return s / len(truth)

### Cross-validation (leave-one-split-out, leak-free). Also produces the out-of-fold round-1 features that round 2 trains on.
The splits overlap: the same training image can be a query in several splits. When split *s* is evaluated, every training row whose query image is also a query in *s* is removed, otherwise the model can memorise those images. This leak inflated CV by about 0.005.

In [10]:
def others(store, s, idx):
    # concat element idx of all splits != s, dropping rows whose query image is a query in split s (leak-free)
    qs = set(store[s][1].qid)
    return pd.concat([store[k][idx][~store[k][idx].qid.isin(qs)] for k in store if k != s])

R1 = {}
for s in VAL:
    Ptr, Qtr = others(VAL, s, 0), others(VAL, s, 1)
    P, Qf = VAL[s][0].copy(), VAL[s][1].copy()
    P["p1"], Qf["pn1"] = fit_predict(Ptr, Qtr, P, Qf, PAIR_F1, Q_F1, [0])
    R1[s] = round2_feats(P, Qf)
    log(f"CV round 1: split {s} done")

cv = {("r1", 1.0): []} | {("r2", w): [] for w in W_GRID}
for s in EVAL_SPLITS:
    P, Qf = R1[s]
    pr1 = rank_preds(P, Qf, "p1", "pn1", 1.0)                      # rank once, then score
    cv[("r1", 1.0)].append(map5([pr1[q] for q in Qf.qid], Qf.truth))
    Ptr, Qtr = others(R1, s, 0), others(R1, s, 1)
    P = P.copy(); Qf = Qf.copy()
    P["p2"], Qf["pn2"] = fit_predict(Ptr, Qtr, P, Qf, PAIR_F2, Q_F2, [0])
    for w in W_GRID:
        pr = rank_preds(P, Qf, "p2", "pn2", w); cv[("r2", w)].append(map5([pr[q] for q in Qf.qid], Qf.truth))
    log(f"CV round 2: split {s} done | r2 MAP@5 " + " ".join(f"w{w}={cv[('r2', w)][-1]:.4f}" for w in W_GRID))
cv = {k: v for k, v in cv.items() if v}
for k, v in cv.items(): print(f"{k[0]} w_new={k[1]:<5} CV MAP@5 {np.mean(v):.4f} ± {np.std(v):.4f}  {np.round(v, 4)}")
W_BEST = max(W_GRID, key=lambda w: np.mean(cv[("r2", w)]))
print("chosen w_new =", W_BEST)

10:17:22 CV round 1: split 0 done


10:17:35 CV round 1: split 1 done


10:17:46 CV round 1: split 2 done


10:17:58 CV round 1: split 3 done


10:18:12 CV round 1: split 4 done


10:18:24 CV round 1: split 5 done


... (15 outputs left out)

10:21:21 CV round 2: split 5 done | r2 MAP@5 w0.8=0.8515 w1.0=0.8512 w1.25=0.8517


10:21:33 CV round 2: split 6 done | r2 MAP@5 w0.8=0.8625 w1.0=0.8640 w1.25=0.8641


10:21:45 CV round 2: split 7 done | r2 MAP@5 w0.8=0.8599 w1.0=0.8592 w1.25=0.8615


r1 w_new=1.0   CV MAP@5 0.8515 ± 0.0048  [0.851  0.8485 0.8483 0.8564 0.8491 0.8439 0.8591 0.8557]
r2 w_new=0.8   CV MAP@5 0.8568 ± 0.0041  [0.8577 0.8549 0.8525 0.8623 0.8532 0.8515 0.8625 0.8599]
r2 w_new=1.0   CV MAP@5 0.8573 ± 0.0043  [0.8586 0.8563 0.8532 0.8622 0.8535 0.8512 0.864  0.8592]
r2 w_new=1.25  CV MAP@5 0.8575 ± 0.0046  [0.8582 0.8577 0.851  0.862  0.8535 0.8517 0.8641 0.8615]
chosen w_new = 1.25


### Stage 3: unified LambdaRank over {candidates + `new_individual`}
A LightGBM ranker sees `new_individual` as one more candidate row in each query's group, carrying the query-level features. It learns the full ordering directly with a MAP objective. Its scores are blended with the two-model scores; the blend weight is chosen by CV.

In [11]:
QCOLS = [c for c in Q_F2 if c != "species"]
RANK_F = PAIR_F2 + ["q_" + c for c in QCOLS] + ["is_new_row"]
RPARAMS = dict(objective="lambdarank", metric="map", eval_at=[5], learning_rate=0.03, num_leaves=31, min_child_samples=40,
               feature_fraction=0.8, bagging_fraction=0.8, bagging_freq=1, lambda_l2=1.0, verbose=-1,
               lambdarank_truncation_level=10, num_threads=N_THREADS)
RANK_ROUNDS, ALPHA_GRID = 800, [0.0, 0.5, 0.7, 1.0]   # 1500 rounds overfit in leak-free CV

def make_groups(P, Qf):
    Qx = Qf[["qid"] + QCOLS].rename(columns={c: "q_" + c for c in QCOLS})
    C = P.merge(Qx, on="qid", how="left"); C["is_new_row"] = 0
    N = Qf[["qid", "species"]].merge(Qx, on="qid"); N["cand"] = NEW; N["is_new_row"] = 1
    for c in PAIR_F2:
        if c not in N: N[c] = np.nan
    D = pd.concat([C, N], ignore_index=True)
    if "truth" in Qf: D["y"] = (D.cand == D.qid.map(dict(zip(Qf.qid, Qf.truth)))).astype(int)
    D["_o"] = D.qid.map({q: i for i, q in enumerate(Qf.qid)})
    return D.sort_values(["_o", "is_new_row"], kind="stable").drop(columns="_o").reset_index(drop=True)

def fit_rank_predict(Dtr, Dte, seeds):
    g = Dtr.groupby("qid", sort=False).size().values; s = np.zeros(len(Dte))
    for sd in seeds:
        m = lgb.train(dict(RPARAMS, seed=sd), lgb.Dataset(Dtr[RANK_F], Dtr.y, group=g), RANK_ROUNDS)
        s += m.predict(Dte[RANK_F]) / len(seeds)
    return s

def blend_preds_scores(D, s_rank, P, Qf, alpha):
    # z-normalise (within query) the ranker score and the logit of the two-model probability, then mix
    D = D.assign(sr=s_rank)
    pm = {(q, c): p for q, c, p in zip(P.qid, P.cand, P.p2)}; pn = dict(zip(Qf.qid, Qf.pn2))
    sp = np.clip([pn[q] if c == NEW else pm[(q, c)] for q, c in zip(D.qid, D.cand)], 1e-6, 1 - 1e-6)
    D["sp"] = np.log(sp / (1 - sp))
    for col in ["sr", "sp"]:
        g = D.groupby("qid")[col]; D[col] = (D[col] - g.transform("mean")) / g.transform("std").clip(1e-6)
    D["s"] = alpha * D.sr + (1 - alpha) * D.sp
    return D

def preds_from(D, bias=0.0):
    # bias > 0 pushes new_individual down (subtracted from the NEW row's blended score)
    s = D.s.values - bias * (D.cand.values == NEW)
    D = D.assign(s_adj=s)
    return {q: list(g.cand.values[np.argsort(-g.s_adj.values)[:5]]) for q, g in D.groupby("qid", sort=False)}

def blend_preds(D, s_rank, P, Qf, alpha, bias=0.0):
    return preds_from(blend_preds_scores(D, s_rank, P, Qf, alpha), bias)

cv3 = {a: [] for a in ALPHA_GRID}; CVD = {}
for s in EVAL_SPLITS:
    qs = set(R1[s][1].qid)
    Dtr = pd.concat([make_groups(*R1[k]) for k in R1 if k != s], ignore_index=True)
    Dtr = Dtr[~Dtr.qid.isin(qs)].reset_index(drop=True); Dv = make_groups(*R1[s])
    sr = fit_rank_predict(Dtr, Dv, [0])
    Ptr, Qtr = others(R1, s, 0), others(R1, s, 1)
    P, Qf = R1[s][0].copy(), R1[s][1].copy()
    P["p2"], Qf["pn2"] = fit_predict(Ptr, Qtr, P, Qf, PAIR_F2, Q_F2, [0])
    CVD[s] = (Dv, sr, P[["qid", "cand", "p2"]], Qf[["qid", "pn2", "truth"]])
    for a in ALPHA_GRID:
        pr = blend_preds(Dv, sr, P, Qf, a); cv3[a].append(map5([pr[q] for q in Qf.qid], Qf.truth))
    log(f"CV ranker: split {s} done | " + " ".join(f"a{a}={cv3[a][-1]:.4f}" for a in ALPHA_GRID))
for a, v in cv3.items(): print(f"ranker blend alpha={a:<4} CV MAP@5 {np.mean(v):.4f} ± {np.std(v):.4f}")
ALPHA = max(ALPHA_GRID, key=lambda a: np.mean(cv3[a]))
print("chosen alpha =", ALPHA, "(0 = two-model only, 1 = ranker only)")

10:22:21 CV ranker: split 0 done | a0.0=0.8586 a0.5=0.8593 a0.7=0.8597 a1.0=0.8570


10:22:59 CV ranker: split 1 done | a0.0=0.8563 a0.5=0.8589 a0.7=0.8589 a1.0=0.8591


10:23:35 CV ranker: split 2 done | a0.0=0.8532 a0.5=0.8536 a0.7=0.8538 a1.0=0.8527


10:24:12 CV ranker: split 3 done | a0.0=0.8622 a0.5=0.8623 a0.7=0.8614 a1.0=0.8608


10:24:50 CV ranker: split 4 done | a0.0=0.8535 a0.5=0.8555 a0.7=0.8557 a1.0=0.8551


10:25:31 CV ranker: split 5 done | a0.0=0.8512 a0.5=0.8523 a0.7=0.8521 a1.0=0.8507


10:26:08 CV ranker: split 6 done | a0.0=0.8640 a0.5=0.8649 a0.7=0.8655 a1.0=0.8640


10:26:44 CV ranker: split 7 done | a0.0=0.8592 a0.5=0.8594 a0.7=0.8585 a1.0=0.8576


ranker blend alpha=0.0  CV MAP@5 0.8573 ± 0.0043
ranker blend alpha=0.5  CV MAP@5 0.8583 ± 0.0040
ranker blend alpha=0.7  CV MAP@5 0.8582 ± 0.0040
ranker blend alpha=1.0  CV MAP@5 0.8571 ± 0.0040
chosen alpha = 0.5 (0 = two-model only, 1 = ranker only)


### Sensitivity to the `new_individual` placement (honest CV)
`bias` is subtracted from the `new_individual` score, so a larger bias puts it first less often. This shows how much CV each setting costs, and the validation rate of `new_individual`-first at bias 0. That rate is the target used for the calibrated test variant.

In [12]:
BIAS_GRID = [-0.25, 0.0, 0.25, 0.5, 0.75, 1.0]
CVB = {s: blend_preds_scores(Dv, sr, P, Qf, ALPHA) for s, (Dv, sr, P, Qf) in CVD.items()}
VAL_NEWFIRST = None
for b in BIAS_GRID:
    m, nf = [], []
    for s, D in CVB.items():
        pr = preds_from(D, b); Qf = CVD[s][3]
        m.append(map5([pr[q] for q in Qf.qid], Qf.truth)); nf.append(np.mean([pr[q][0] == NEW for q in Qf.qid]))
    if b == 0.0: VAL_NEWFIRST = float(np.mean(nf))
    print(f"bias {b:+.2f}: CV MAP@5 {np.mean(m):.4f} | new_individual first {np.mean(nf):.3f} | true new {np.mean([np.mean(CVD[s][3].truth == NEW) for s in CVD]):.3f}")
print("validation new_individual-first rate at bias 0:", round(VAL_NEWFIRST, 3))

# calibrated label probabilities (for blending with another model): softmax over {candidates, new} of
# log P(match) and log(w * P(new)), temperature fitted by NLL of the true label on the honest CV OOF.
# Monotone in the two-model scores, so the ranking equals rank_preds(..., W_BEST).
from scipy.optimize import minimize_scalar
def label_logscores(P, Qf, w):
    pn = dict(zip(Qf.qid, Qf.pn2)); out = {}
    for q, g in P.groupby("qid", sort=False):
        out[q] = (list(g.cand.values) + [NEW], np.log(np.clip(np.r_[g.p2.values, w * pn[q]], 1e-6, None)))
    return out
def to_probs(L, T):
    out = {}
    for q, (labs, sc) in L.items():
        z = sc / T; e = np.exp(z - z.max()); out[q] = dict(zip(labs, e / e.sum()))
    return out
CAL = []
for s, (Dv, sr, P, Qf) in CVD.items():
    L = label_logscores(P, Qf, W_BEST); t = dict(zip(Qf.qid, Qf.truth))
    CAL += [(sc, labs.index(t[q])) for q, (labs, sc) in L.items() if t[q] in labs]
def nll(T): return -np.mean([(sc[i] / T) - np.log(np.exp(sc / T - (sc / T).max()).sum()) - (sc / T).max() for sc, i in CAL])
T_CAL = minimize_scalar(nll, bounds=(0.05, 20), method="bounded").x
print(f"probability temperature T = {T_CAL:.3f} (fitted on {len(CAL)} CV queries whose true label is a candidate)")
# honest out-of-fold validation probabilities (splits 0-7) + truth, for tuning a blend with another pipeline
CV_PROBS = {s: (to_probs(label_logscores(P, Qf, W_BEST), T_CAL), dict(zip(Qf.qid, Qf.truth))) for s, (Dv, sr, P, Qf) in CVD.items()}
pickle.dump(CV_PROBS, open(f"{WORK}/cv_probs.pkl", "wb"))
print("saved cv_probs.pkl:", {s: len(v[0]) for s, v in CV_PROBS.items()})

bias -0.25: CV MAP@5 0.8579 | new_individual first 0.322 | true new 0.224


bias +0.00: CV MAP@5 0.8583 | new_individual first 0.295 | true new 0.224


bias +0.25: CV MAP@5 0.8574 | new_individual first 0.268 | true new 0.224


bias +0.50: CV MAP@5 0.8550 | new_individual first 0.244 | true new 0.224


bias +0.75: CV MAP@5 0.8513 | new_individual first 0.221 | true new 0.224


bias +1.00: CV MAP@5 0.8448 | new_individual first 0.198 | true new 0.224
validation new_individual-first rate at bias 0: 0.295


probability temperature T = 1.072 (fitted on 18353 CV queries whose true label is a candidate)


saved cv_probs.pkl: {0: 2525, 1: 2406, 2: 2434, 3: 2439, 4: 2342, 5: 2311, 6: 2452, 7: 2456}


## 6. Final model on test

In [13]:
Pall = pd.concat([VAL[k][0] for k in VAL]); Qall = pd.concat([VAL[k][1] for k in VAL])
PT1, QT1 = PT.copy(), QT.copy()
M1, M2 = [], []                                                                  # fitted models (reused for self-training)
PT1["p1"], QT1["pn1"] = fit_predict(Pall, Qall, PT1, QT1, PAIR_F1, Q_F1, LGB_SEEDS, M1)       # round 1 on all splits
log("final: round 1 fitted on all splits")
PT2, QT2 = round2_feats(PT1, QT1)
P2all = pd.concat([R1[k][0] for k in R1]); Q2all = pd.concat([R1[k][1] for k in R1])
PT2["p2"], QT2["pn2"] = fit_predict(P2all, Q2all, PT2, QT2, PAIR_F2, Q_F2, LGB_SEEDS, M2)     # round 2 on OOF round-1 feats
log("final: round 2 fitted")
# free everything the ranker does not need (this step is the memory peak on a 16 GB machine)
del Pall, Qall, P2all, Q2all, PT1, QT1, VAL, CVB, CVD; gc.collect()
def slim(D):
    D = D[["qid", "y"] + RANK_F].copy(); f = D.select_dtypes("float64").columns
    D[f] = D[f].astype(np.float32); return D
DT = make_groups(PT2, QT2)
DR = pd.concat([slim(make_groups(*R1[k])) for k in R1], ignore_index=True); gc.collect()
log(f"final: ranker train table {DR.shape}, {DR.memory_usage().sum()/2**20:.0f} MB")
SR = fit_rank_predict(DR, DT, LGB_SEEDS); del DR; gc.collect()
log("final: ranker fitted")
DTS = blend_preds_scores(DT, SR, PT2, QT2, ALPHA)
pickle.dump(DTS[["qid", "cand", "s"]], open(f"{WORK}/test_scores.pkl", "wb"))
pred = preds_from(DTS, 0.0)
sub = pd.DataFrame({"id": test.id, "predictions": [" ".join(pred[i]) for i in test.id]})
# calibrated variant: smallest bias at which test new_individual-first rate <= validation rate (covariate-shift correction)
def newfirst(pr): return np.mean([pr[q][0] == NEW for q in test.id])
BIAS_CAL = next(b for b in np.arange(0, 3.01, 0.01) if newfirst(preds_from(DTS, b)) <= VAL_NEWFIRST)
pred_cal = preds_from(DTS, BIAS_CAL)
sub_cal = pd.DataFrame({"id": test.id, "predictions": [" ".join(pred_cal[i]) for i in test.id]})
print(f"test new_individual-first: default {newfirst(pred):.3f} -> calibrated {newfirst(pred_cal):.3f} (bias {BIAS_CAL:.2f}, val rate {VAL_NEWFIRST:.3f})")
TEST_PROBS = to_probs(label_logscores(PT2, QT2, W_BEST), T_CAL)
assert set(TEST_PROBS) == set(test.id) and all(abs(sum(d.values()) - 1) < 1e-6 for d in TEST_PROBS.values())
pickle.dump(TEST_PROBS, open(f"{WORK}/test_probs.pkl", "wb"))
print(f"saved test_probs.pkl: {len(TEST_PROBS)} test queries, mean P(new) {np.mean([d[NEW] for d in TEST_PROBS.values()]):.3f}")

10:27:44 final: round 1 fitted on all splits


10:28:35 final: round 2 fitted


10:28:37 final: ranker train table (818958, 70), 241 MB


10:30:41 final: ranker fitted


test new_individual-first: default 0.390 -> calibrated 0.294 (bias 0.93, val rate 0.295)


saved test_probs.pkl: 2791 test queries, mean P(new) 0.333


## 6b. Self-training: pseudo-label gallery expansion
Test images whose top identity is very confident (round-2 P(match) >= 0.9 and P(new) <= 0.1) are added to the gallery
with their **predicted** identity (no human or LLM labels; standard semi-supervised self-training). The remaining test
images are then re-scored by the same fitted models against the expanded gallery: MiewID retrieval, SIFT verification and
LNBNN keypoint matching can now also use other test photos of the same animal. Simulated on the honest validation splits
(round-1 model, 8 splits): pseudo-label precision ~0.99, MAP@5 +0.0025 on average, better on 7/8 splits.
Confident images keep their first predictions. **On the public leaderboard it scored lower (0.85829 vs 0.86164), most
likely because the test has more new animals (~31 % vs 22 %), so it is switched off for the final submission.**

In [14]:
SELF_TRAIN = False   # tried: public LB 0.85829 with it vs 0.86164 without -> OFF for the final submission
if not SELF_TRAIN:
    print("self-training skipped (SELF_TRAIN = False)")
else:
    PSEUDO_P, PSEUDO_N = 0.9, 0.1
    top = PT2.sort_values("p2", ascending=False).drop_duplicates("qid").set_index("qid")
    pn2 = QT2.set_index("qid").pn2
    CONF = top[(top.p2 >= PSEUDO_P) & (pn2.reindex(top.index) <= PSEUDO_N)].cand
    log(f"self-training: {len(CONF)} confident test images ({len(CONF) / len(test):.2f}) join the gallery")
    SIFT = load_sift(SIFT_TR, train.id); SIFT.update(load_sift(SIFT_TE, test.id)); SC = SiftCache()   # freed earlier

    class LNBExpanded:
        # LNBNN over the stored train neighbour lists merged with an exact index of the pseudo-labelled test descriptors
        def __init__(self, base, extra_ids, extra_labels, query_ids):
            self.b, self.gal, self.names = base, base.gal, base.names
            self.epos = {g: k for k, g in enumerate(extra_ids)}; self.ext = {}
            if not len(extra_ids): return
            X = [root_desc(g).reshape(-1, 128) for g in extra_ids]
            owner = np.repeat(np.arange(len(extra_ids)), [len(x) for x in X]).astype(np.int32)
            index = faiss.IndexFlatL2(128); index.add(np.concatenate(X).astype(np.float32))
            self.elab = np.searchsorted(base.names, np.asarray(extra_labels)).astype(np.int64)
            for b0 in range(0, len(query_ids), 200):
                qs = list(query_ids[b0:b0 + 200]); Q = [root_desc(q).reshape(-1, 128) for q in qs]
                if not sum(len(x) for x in Q): continue
                d, i = index.search(np.concatenate(Q).astype(np.float32), LNB_KNN); o = 0
                for q, x in zip(qs, Q):
                    n = len(x)
                    if n: self.ext[q] = (owner[np.maximum(i[o:o + n], 0)], np.sqrt(np.maximum(d[o:o + n], 0)), i[o:o + n] >= 0)
                    o += n
        def scores(self, qid, gmask):
            b = self.b; i = b.qpos[qid]; a, e = b.qoff[i], b.qoff[i + 1]
            if e == a or qid not in self.ext: return b.scores(qid, gmask)
            nb = np.asarray(b.NB[a:e]); d = np.asarray(b.DI[a:e], np.float32)
            ok = (nb >= 0) & gmask[np.maximum(nb, 0)]
            if qid in b.gpos: ok &= nb != b.gpos[qid]
            enb, ed, eok = self.ext[qid]
            if qid in self.epos: eok = eok & (enb != self.epos[qid])                 # never match the query image itself
            lab = np.concatenate([np.where(ok, b.lab[np.maximum(nb, 0)], -1), np.where(eok, self.elab[enb], -1)], 1)
            d = np.concatenate([d, ed.astype(np.float32)], 1); okk = lab >= 0
            o = np.argsort(np.where(okk, d, np.inf), 1, kind="stable")
            d, lab, okk = (np.take_along_axis(x, o, 1) for x in (d, lab, okk))
            rank = np.cumsum(okk, 1); has_bg = rank[:, -1] >= LNB_K + 1
            bg = np.where(has_bg, d[np.arange(len(d)), np.argmax(rank >= LNB_K + 1, 1)], np.max(np.where(okk, d, 0), 1))
            r, c = np.nonzero(okk & (rank <= LNB_K)); out = np.zeros(len(b.names), np.float32)
            _, first = np.unique(r.astype(np.int64) * len(b.names) + lab[r, c], return_index=True); r, c = r[first], c[first]
            np.add.at(out, lab[r, c], np.maximum(bg[r] - d[r, c], 0)); return out

    t = time.time()
    cols = ["id", "species", "viewpoint", "identity"]
    tpos = {i: k for k, i in enumerate(test.id)}
    Gx = pd.concat([train[cols], test.set_index("id").loc[list(CONF.index)].rename_axis("id").reset_index()
                    .assign(identity=CONF.values)[cols]], ignore_index=True)
    Xgx = np.concatenate([Xtr, Xte[[tpos[i] for i in CONF.index]]])
    LNB_BASE = dict(LNB)
    for sp in ["hyena", "leopard"]:
        ids = [q for q in CONF.index if (test.species.values[tpos[q]] == sp)]
        LNB[sp] = LNBExpanded(LNB_BASE[sp], ids, CONF[ids].values, test.id[test.species == sp].values)
    PTx, QTx = build(Gx, Xgx, test, Xte)
    LNB.update(LNB_BASE); del SIFT, SC; gc.collect()
    for df in (PTx, QTx):
        for c in df.select_dtypes("float64").columns: df[c] = df[c].astype(np.float32)
    PTx["p1"], QTx["pn1"] = predict_with(M1, PTx, QTx, PAIR_F1, Q_F1)
    PTx2, QTx2 = round2_feats(PTx, QTx)
    PTx2["p2"], QTx2["pn2"] = predict_with(M2, PTx2, QTx2, PAIR_F2, Q_F2)
    PX = to_probs(label_logscores(PTx2, QTx2, W_BEST), T_CAL)
    TEST_PROBS_X = {q: (TEST_PROBS[q] if q in CONF.index else PX[q]) for q in test.id}
    assert set(TEST_PROBS_X) == set(test.id) and all(abs(sum(d.values()) - 1) < 1e-6 for d in TEST_PROBS_X.values())
    pickle.dump(TEST_PROBS_X, open(f"{WORK}/test_probs_pseudo.pkl", "wb"))
    t1 = lambda D, q: max(D[q], key=D[q].get)
    print(f"saved test_probs_pseudo.pkl | top-1 changed on {np.mean([t1(TEST_PROBS, q) != t1(TEST_PROBS_X, q) for q in test.id]):.3f} "
          f"of test images | {time.time() - t:.0f}s")

self-training skipped (SELF_TRAIN = False)


## 7. Submission checks

In [15]:
allowed = {s: set(train.identity[train.species == s]) | {NEW} for s in ["hyena", "leopard"]}
def check_and_save(sub, path):
    assert list(sub.columns) == ["id", "predictions"]
    assert len(sub) == len(test) == 2791 and sub.id.is_unique and set(sub.id) == set(test.id)
    for i, s, p in zip(test.id, test.species, sub.predictions):
        labs = p.split(" ")
        assert len(labs) == 5 and len(set(labs)) == 5 and all(labs), i   # 5 distinct labels, single spaces
        assert set(labs) <= allowed[s], (i, labs)                        # same-species identity or new_individual
    sub.to_csv(path, index=False); assert pd.read_csv(path).equals(sub)
check_and_save(sub, f"{WORK}/submission.csv")
check_and_save(sub_cal, f"{WORK}/submission_newcal.csv")
print("saved submission_newcal.csv | top-1 differs from submission.csv on",
      f"{np.mean(sub.predictions.str.split().str[0] != sub_cal.predictions.str.split().str[0]):.3f} of rows")
first = sub.predictions.str.split().str[0]
print(f"saved {WORK}/submission.csv | rows {len(sub)} | new_individual first {np.mean(first==NEW):.3f} "
      f"| CV MAP@5 (final blend, alpha={ALPHA}) {np.mean(cv3[ALPHA]):.4f}")
for s in ["hyena", "leopard"]:
    m = (test.species == s).values; print(f"  {s}: new_individual first {np.mean(first[m]==NEW):.3f}")
sub.head(10)

saved submission_newcal.csv | top-1 differs from submission.csv on 0.096 of rows
saved ./submission.csv | rows 2791 | new_individual first 0.390 | CV MAP@5 (final blend, alpha=0.5) 0.8583
  hyena: new_individual first 0.402
  leopard: new_individual first 0.381


,id,predictions
0,te_00001,LEO0607 new_individual LEO0654 LEO0546 LEO0605
1,te_00002,new_individual HYE0111 HYE0286 HYE0662 HYE0524
2,te_00003,LEO0189 new_individual LEO0674 LEO0172 LEO0131
3,te_00004,LEO0035 LEO0321 new_individual LEO0502 LEO0509
4,te_00005,new_individual LEO0318 LEO0043 LEO0003 LEO0605
5,te_00006,LEO0494 new_individual LEO0015 LEO0321 LEO0260
6,te_00007,LEO0373 new_individual LEO0509 LEO0087 LEO0100
7,te_00008,new_individual HYE0411 HYE0293 HYE0248 HYE0166
8,te_00009,new_individual LEO0345 LEO0620 LEO0040 LEO0396
9,te_00010,new_individual LEO0637 LEO0313 LEO0003 LEO0422
